In [1]:
from data_code.dataset_builder import build_dataset

import torch
import torch.nn as nn

In [2]:
int_emb = torch.load('data_code/INT_embeddings.pt')
rt_emb = torch.load('data_code/RT_embeddings.pt')
rn_emb = torch.load('data_code/RNaseH_embeddings.pt')
neg_emb = torch.load('data_code/negatives_embeddings.pt')
class_embeddings = {"INT": int_emb, "RT": rt_emb, "RN": rn_emb, "NEG": neg_emb}

train_loader, val_loader, class_weights, label_map = build_dataset(class_embeddings = class_embeddings, window_size= 128)

Label map: {'INT': 0, 'RT': 1, 'RN': 2, 'NEG': 3}
Train: 55, Val: 14
Train distribution: {'INT': 3, 'RT': 3, 'RN': 3, 'NEG': 46}
Class weights: {'INT': 1.305, 'RT': 1.305, 'RN': 1.305, 'NEG': 0.085}


In [3]:
class BioSequenceClassifier(nn.Module):
    def __init__(self):
        super(BioSequenceClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        return self.model(x)

In [7]:
device = 'cuda'

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
model = BioSequenceClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01, weight_decay= 1e-5)
epochs = 40

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    num_samples = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        num_samples += batch_size

    epoch_loss = running_loss / num_samples

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
                
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
    val_epoch_loss = val_loss / len(val_loader.dataset)
    val_acc = 100 * correct / total
        
    history['train_loss'].append(epoch_loss)
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_acc)
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | "
                f"Train Loss: {epoch_loss:.4f} | "
                f"Val Loss: {val_epoch_loss:.4f} | "
                f"Val Acc: {val_acc:.2f}%")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)